# Research notebook

Read the module README before executing. Textual outputs below are saved historical snapshots; the report follows the original project document and was not recalculated during publication cleanup. Setup paths have been made portable. Embedded media and machine-specific diagnostic output are omitted from this public-facing copy.


In [ ]:
# Portable project paths; this cell does not load a model.
from pathlib import Path
import sys

_candidates = (Path.cwd(), *Path.cwd().parents)
_project_root = next((p for p in _candidates if (p / "project_paths.py").is_file()), None)
if _project_root is None:
    raise RuntimeError("Start Jupyter from the retrieval repository or one of its subdirectories.")
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
from project_paths import (GALLERY_DIR, SIGLIP_MODEL, REFERENCE_CLASS_FOLDERS,
                           reference_class_indices)
# Experiment settings. Edit REFERENCE_CLASS_FOLDERS in project_paths.py for your data.
target_classes = list(REFERENCE_CLASS_FOLDERS)
CLIP_MODEL = "ViT-B/32"
CLIP_LARGE_MODEL = "openai/clip-vit-large-patch14"
REFERENCE_COUNTS = [1, 5, 10, 20]
IMAGE_ENCODER = 2  # 0: CLIP B/32; 1: CLIP L/14; 2: SigLIP



In [2]:
import os
from PIL import Image
import torch
import clip
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load(CLIP_MODEL, device=device)
import numpy as np
from pathlib import Path

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset
import random
from transformers import BertTokenizer, BertForSequenceClassification, CLIPProcessor, CLIPModel
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel, BitsAndBytesConfig
import accelerate
import sentencepiece

In [4]:
#区分数据集为参考图和图片池
def split(dataset, target_label, ref_number, seed=0):
    #获取所有图片索引，在目标类别选取参考图，再分开数据集
    all_indices=list(range(len(dataset)))
    target_indices=[idx for idx in all_indices if dataset[idx][1]==target_label]
    random.seed(seed)
    ref_indices=random.sample(target_indices, ref_number)
    gallery_indices=[idx for idx in all_indices if idx not in ref_indices]
    return Subset(dataset,ref_indices), Subset(dataset,gallery_indices)



In [5]:

def f1_score(similarity,gallery_label,threshold,target_label):
    #pos是布尔值，在simliarity中使用可以获得目标标签的变量
    pos_bool=(gallery_label==target_label)
    neg_bool=(gallery_label!=target_label)
    pos=similarity[pos_bool]
    neg=similarity[neg_bool]
    TP=sum(pos>=threshold)
    FP=sum(neg>=threshold)
    FN=sum(pos<threshold)
    precision=TP/(TP+FP+0.0001)
    recall=TP/(TP+FN+0.0001)
    f1=2*precision*recall/(precision+recall+0.0001)
    return f1,precision,recall
def find_threshold(similarity,gallery_label,target_label):
    thresholds=np.linspace(np.min(similarity),np.max(similarity),100)
    best_f1=0
    best_precision=0
    best_recall=0
    for threshold in thresholds:
        f1,precision,recall=f1_score(similarity,gallery_label,threshold,target_label)
        if f1 > best_f1:
            best_f1=f1
            best_precision=precision
            best_recall=recall
    return best_f1,best_precision,best_recall




In [ ]:
def main():
 image_encoder=IMAGE_ENCODER
 if image_encoder==0:
    model, preprocess = clip.load(CLIP_MODEL, device=device)
    dataset=ImageFolder(root=str(GALLERY_DIR),transform=preprocess)
 if image_encoder==1:
    model = CLIPModel.from_pretrained(CLIP_LARGE_MODEL).to(device).eval()
    processor = CLIPProcessor.from_pretrained(CLIP_LARGE_MODEL)
    print('模型加载完成')
    def clip_transform(image):
    # 1. 用processor处理图像，得到字典
        inputs = processor(images=image, return_tensors="pt")
    # 2. 提取pixel_values并移除batch维度（从[1,3,224,224]转为[3,224,224]）
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values
    dataset=ImageFolder(root=str(GALLERY_DIR),transform=clip_transform)
    print('数据集加载完成')
 if image_encoder==2:
     model = AutoModel.from_pretrained(SIGLIP_MODEL).to(device).eval()
     processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
     print('模型加载完成')
     def clip_transform(image):
    # 1. 用processor处理图像，得到字典
        inputs = processor(images=image, return_tensors="pt")
    # 2. 提取pixel_values并移除batch维度（从[1,3,224,224]转为[3,224,224]）
        pixel_values = inputs["pixel_values"].squeeze(0)
        return pixel_values
     dataset=ImageFolder(root=str(GALLERY_DIR),transform=clip_transform)
     print('数据集加载完成')

 for target_class in target_classes:
    print(target_class)
    for ref_number in REFERENCE_COUNTS:
        dic = reference_class_indices(dataset.class_to_idx)
        #target_class='Dog'
        #ref_number=10
        target_label=dic[target_class]
        #使用imageFolder导入数据集
        ref_dataset,gallery_dataset=split(dataset,target_label,ref_number)
        ref = torch.utils.data.DataLoader(ref_dataset,shuffle=False)
        gallery=torch.utils.data.DataLoader(gallery_dataset, batch_size=256, num_workers=0, shuffle=False)
        print('数据集创建完成')
        model.eval()
        gallery_features=[]
        ref_features=[]
        with torch.no_grad():
         if image_encoder==0:
            for images, _ in gallery:
                images = images.to(device)
                batch_features = model.encode_image(images)
                gallery_features.append(batch_features)
            #        处理参考图像中的所有批次
            for images, _ in ref:
                images = images.to(device)
                batch_features = model.encode_image(images)
                ref_features.append(batch_features)
         if image_encoder == 1 or image_encoder == 2:
            for pixel_values, _ in gallery:  # 直接获取pixel_values（已由clip_transform处理）
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                gallery_features.append(batch_features)
            print('特征创建完成')
            for pixel_values, _ in ref:
                pixel_values = pixel_values.to(device)
                batch_features = model.get_image_features(pixel_values=pixel_values)  # 正确参数
                batch_features = batch_features / batch_features.norm(dim=1, keepdim=True)
                ref_features.append(batch_features)
            print('特征创建完成')
    # 合并所有批次的特征
        gallery_features = torch.cat(gallery_features, dim=0)
        ref_features = torch.cat(ref_features, dim=0)
    # 多张参考图特征求平均
        ref_features=ref_features.mean(dim=0, keepdim=True)
    # 计算相似度矩阵
        similarity = gallery_features @ ref_features.T
        similarity=  similarity.cpu().numpy()
        gallery_label = [sample[1] for sample in gallery_dataset]  # sample[1]是标签
        gallery_label = np.array(gallery_label)
        best_f1,best_precision,best_recall=find_threshold(similarity,gallery_label,target_label)
        print(f'{target_class},{ref_number}:f1:{best_f1},precision:{best_precision},recall:{best_recall}')


In [9]:
main()

模型加载完成
数据集加载完成
Dog
数据集创建完成
特征创建完成
特征创建完成
Dog,1:f1:[0.57353779],precision:[0.45921436],recall:[0.76381871]
数据集创建完成
特征创建完成
特征创建完成
Dog,5:f1:[0.733618],precision:[0.71921147],recall:[0.74871756]
数据集创建完成
特征创建完成
特征创建完成
Dog,10:f1:[0.71834082],precision:[0.79113874],recall:[0.65789439]
数据集创建完成
特征创建完成
特征创建完成
Dog,20:f1:[0.72387648],precision:[0.80821862],recall:[0.65555519]
Duck
数据集创建完成
特征创建完成
特征创建完成
Duck,1:f1:[0.69669151],precision:[0.69499965],recall:[0.69849211]
数据集创建完成
特征创建完成
特征创建完成
Duck,5:f1:[0.7704416],precision:[0.82456092],recall:[0.72307655]
数据集创建完成
特征创建完成
特征创建完成
Duck,10:f1:[0.77501738],precision:[0.79888224],recall:[0.75263118]
数据集创建完成
特征创建完成
特征创建完成
Duck,20:f1:[0.79660694],precision:[0.79888224],recall:[0.794444]
Erhu
数据集创建完成
特征创建完成
特征创建完成
Erhu,1:f1:[0.71096991],precision:[0.60638276],recall:[0.85929605]
数据集创建完成
特征创建完成
特征创建完成
Erhu,5:f1:[0.67450861],precision:[0.54807675],recall:[0.87692263]
数据集创建完成
特征创建完成
特征创建完成
Erhu,10:f1:[0.70152032],precision:[0.6979163],recall:[0.70526279]
数据集创建完成
